# 12 · Reprodução de Lira et al.: Tabelas 4 e 5 com e sem vazamento (59 arquiteturas) · aba B

Roda a receita escolhida no 11 (`resultados/calibracao_lira.json`) nas 59 arquiteturas das Tabelas 4 e 5, nas duas tarefas e em duas partições:

* `vazado_p42`: `iterative_split` do leguwoi (o hold-out do código público, com casos em treino e teste);
* `agrupado_d0`: dobra 0 de `folds_entrada_saida.csv` como teste (~80/20, nenhum caso dos dois lados).

**Só a partição muda.** Modelos, aumentações, reamostragem, otimizador, épocas e lote são os mesmos nos dois lados.

A fila roda primeiro as 12 arquiteturas da dissertação e depois as outras 47 (236 unidades no total). Cada unidade grava um `resultado.json` e é pulada se já existir; unidades longas gravam checkpoint a cada 30 min e retomam após queda. Esta cópia percorre a fila **de trás para frente**, para rodar em paralelo com o `12_reproducao_lira_tabelas.ipynb` em outra aba (as travas evitam trabalho repetido).

**Como rodar:** **Executar tudo**; se cair, executar tudo de novo. Qualquer GPU serve; use **RAM alta** se a receita tiver SMOTEENN.

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"
RAIZ_TRABALHO = "/content"
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_MODULO_ESPERADO = "ed0ede175dc0827c20dfc804ba69fa63740509ff576631999193597fa09ba004"   # reproducao_lira.py
ABA = "B"
INVERTER_FILA = True
SO_OS_DOZE = False           # True = para depois das 12 arquiteturas da dissertação
PARTICOES = ("vazado_p42", "agrupado_d0")   # repetições depois: ("vazado_p43", "agrupado_d1"), ...
CKPT_MIN = 30.0

In [ ]:
# Versões do environment.yml do leguwoi onde o Colab permite; o que não instalar fica na versão do
# Colab e isso é registrado em cada resultado.json (campo "ambiente"). Rode ANTES de importar.
import subprocess, sys
def pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
pip("scikit-multilearn==0.2.0")
FIXOS = {
    "albumentations": ("1.4.0", "import numpy as np, albumentations as A; from albumentations.pytorch import ToTensorV2; "
        "A.Compose([A.RGBShift(p=1), A.UnsharpMask(p=1), A.SafeRotate(limit=30, p=1), A.GaussNoise(p=1), A.ISONoise(p=1), "
        "A.Sharpen(p=1), A.CLAHE(p=1), A.ElasticTransform(p=1), A.Resize(224, 224), A.Normalize(), ToTensorV2()])"
        "(image=np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8))"),
    "imbalanced-learn": ("0.12.0", "import numpy as np; from imblearn.combine import SMOTEENN; "
        "SMOTEENN(random_state=64).fit_resample(np.random.rand(80, 6), np.array([0] * 60 + [1] * 20))"),
    "skorch": ("0.15.0", "import numpy as np, torch, skorch; from skorch import NeuralNetClassifier; "
        "NeuralNetClassifier(torch.nn.Linear, module__in_features=4, module__out_features=2, max_epochs=1, verbose=0, "
        "criterion=torch.nn.CrossEntropyLoss, train_split=skorch.dataset.ValidSplit(cv=5, stratified=True))"
        ".fit(np.random.rand(40, 4).astype('float32'), np.array([0, 1] * 20))"),
}
for pacote, (versao, teste) in FIXOS.items():
    ok = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{pacote}=={versao}"]).returncode == 0
    ok = ok and subprocess.run([sys.executable, "-c", teste], capture_output=True).returncode == 0
    if not ok:
        print(f"⚠️ {pacote}=={versao} não funciona neste Colab; instalando a versão atual (fica registrado).")
        pip("-U", pacote)
        r = subprocess.run([sys.executable, "-c", teste], capture_output=True, text=True)
        assert r.returncode == 0, f"{pacote} não funciona nem na versão atual:\n{r.stderr[-800:]}"
    else:
        print(f"{pacote}=={versao} OK")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, "O protocolo_comum.py do Drive não é o publicado."
ARQ_MODULO = f"{DIR_DRIVE}/reproducao_lira.py"
assert pc.sha256_arquivo(ARQ_MODULO) == SHA_MODULO_ESPERADO, (
    "O reproducao_lira.py do Drive não é o do repositório (SHA diferente). Envie de novo o arquivo certo.")
import reproducao_lira as rl
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução → Alterar tipo → GPU (qualquer uma serve aqui)."
pc.definir_id_sessao(ABA)
print("GPU:", torch.cuda.get_device_name(0), "| módulo", rl.VERSAO, SHA_MODULO_ESPERADO[:16], "… | dono das travas:", pc.ID_SESSAO)

# Clona dataset e leguwoi nos commits fixados, importa o código do leguwoi e confere as sementes dele
ctx = rl.preparar(RAIZ_TRABALHO, caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}")
print("Sementes no leguwoi:", *ctx.meta["sementes_no_leguwoi"], sep="\n  ")
print("V_0.1 × master:", json.dumps(ctx.meta["diferencas_v01"], indent=1))
erros = rl.testar_metricas()
assert not erros, erros
print("Partição com vazamento = a do 04b:", rl.conferir_com_holdout_do_drive(ctx, f"{DIR_DRIVE}/dados/holdout_lira2024.csv"))
display(pd.DataFrame([rl.resumo_particao(ctx, t, p) for t in rl.TAREFAS for p in ("vazado_p42", "agrupado_d0")]))
print("Ambiente:", rl.ambiente())

R = f"{DIR_DRIVE}/resultados/reproducao_lira"
C = f"{DIR_DRIVE}/checkpoints/reproducao_lira"
R_CAL = f"{DIR_DRIVE}/resultados/reproducao_lira_calibracao"
dec = json.loads((Path(DIR_DRIVE) / "resultados" / "calibracao_lira.json").read_text())
assert dec["modulo_sha256"] == SHA_MODULO_ESPERADO, "A calibração foi feita com outra versão do reproducao_lira.py."
RECEITA, PRECISAO = dec["receita"], dec["precisao"]
KW = {"epocas": dec["epocas_B"]} if RECEITA != "A" else {}
print("Receita:", RECEITA, rl.RECEITAS[RECEITA], "| precisão:", PRECISAO, "|", KW)

# aproveita as unidades da calibração que são idênticas às desta fila
for arq in Path(R_CAL).glob(f"{RECEITA}/vazado_p42/*/*/resultado.json"):
    destino = Path(R) / arq.relative_to(R_CAL)
    r = json.loads(arq.read_text())
    if not destino.exists() and r["precisao"] == PRECISAO and r["epocas"] == KW.get("epocas", rl.RECEITAS[RECEITA]["epocas"]):
        destino.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(arq, destino); print("aproveitado da calibração:", destino.parent.name, destino.parent.parent.name)

## Rodada

In [ ]:
fila = rl.montar_fila(RECEITA, modelos=rl.DOZE if SO_OS_DOZE else None, particoes=PARTICOES)
if INVERTER_FILA:
    fila = fila[::-1]
print(len(fila), "unidades na fila")
feitos = rl.executar_fila(ctx, fila, R, C, precisao=PRECISAO, ckpt_min=CKPT_MIN, **KW)
print("Concluídas nesta sessão:", len(feitos))

## Andamento

In [ ]:
res = rl.ler_resultados(R, RECEITA)
print(len(res), "de", len(rl.MODELOS) * 2 * len(PARTICOES), "unidades prontas |", f"{res['treino_min'].sum() / 60:.1f} h de treino até aqui")
display(res.pivot_table(index="modelo", columns=["tarefa", "particao"], values="acc").reindex(rl.MODELOS).round(3))
erros = sorted(Path(R).glob("*/*/*/*/erro_*.json"))
if erros:
    print("Unidades com erro registrado:"); [print(" ", e.parent.relative_to(R), json.loads(e.read_text())["erro"][:150]) for e in erros]

⚠️ Ao terminar, desconecte o ambiente (*Ambiente de execução → Desconectar e excluir*).